In [1]:
%load_ext autoreload
%autoreload 2

import random
import sys
from ast import literal_eval
from urllib import parse

import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

sys.path.append('..')


/home/michel.yildirim/.conda/envs/jupyter_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def fix_title(title):
    return parse.unquote(title).replace('_', ' ')

# Prepare ranking functions

In [16]:
def model_best_candidate(model, prediction_head, tokenizer, contexts, source_sections, target_title, target_lead,
                         mentions):
    best_context = {'score': float('-inf'), 'context': ''}
    all_scores = []

    if target_title not in mentions:
        mentions[target_title] = ''

    for context, source_section in zip(contexts, source_sections):
        input = ["", ""]
        input[0] = f"{target_title} {mentions[target_title]}{tokenizer.sep_token}{target_lead}"
        input[1] = f"{source_section}{tokenizer.sep_token}{context}"
        input_tokens = tokenizer([input], return_tensors='pt', padding='max_length', truncation=True,
                                 max_length=512).to(device)
        embeddings = model(**input_tokens)['last_hidden_state'][:, 0, :]
        prediction = prediction_head(embeddings).squeeze().detach().cpu().numpy()
        
        all_scores.append({'score': prediction.item(), 'context': context})
        if prediction.item() > best_context['score']:
            best_context['score'] = prediction.item()
            best_context['context'] = context

    return best_context, all_scores

# Load model

In [4]:
device = 'cuda'

model = AutoModel.from_pretrained('/home/aarora/multilingual_entity_insertion/models/roberta_full_multilingual-all/model')
tokenizer = AutoTokenizer.from_pretrained('/home/aarora/multilingual_entity_insertion/models/roberta_full_multilingual-all/tokenizer')
prediction_head = nn.Sequential(nn.Linear(model.config.hidden_size, model.config.hidden_size),
                                nn.ReLU(),
                                nn.Linear(model.config.hidden_size, 1))
prediction_head.load_state_dict(
    torch.load('/home/aarora/multilingual_entity_insertion/models/roberta_full_multilingual-all/classification_head.pth', map_location='cpu'))

model = model.to(device)
prediction_head = prediction_head.to(device)

model = model.eval()
prediction_head = prediction_head.eval()

# Load and prepare data

In [5]:
df = pd.read_parquet('/home/nikolaj.mikkelsen/locei_dataset/test/en.parquet')
mentions = pd.read_parquet('/home/nikolaj.mikkelsen/locei_dataset/test/en_mention_map.parquet')

In [6]:
df['target_title'] = df['target_title'].apply(fix_title)

In [8]:
sample = df.sample(n=3, random_state=42)

target_title = sample['target_title']
target_lead = sample['target_lead']
first_version = sample['first_version']
print(target_title)
print(target_lead)
print(first_version)

521        Quillette
737    Republicanism
740        Pop Trash
Name: target_title, dtype: object
521    Quillette is an online magazine founded by Aus...
737    Republicanism is a political ideology centred ...
740    Pop Trash is the tenth studio album by English...
Name: target_lead, dtype: object
521    1179676986
737    1179279064
740    1176838804
Name: first_version, dtype: int64


In [9]:
mention_map_pre = mentions.to_dict('records')
mention_map = {}
for row in mention_map_pre:
    title = fix_title(row['target_title'])
    if title in mention_map:
        mention_map[title].append(row['mention'])
    else:
        mention_map[title] = [row['mention']]

for title in mention_map:
    mention_map[title] = list(set([mention.lower() for mention in mention_map[title]]))
    if len(mention_map[title]) > 10:
        mention_map[title].sort(key=lambda x: len(x))
        while len(mention_map[title]) > 10 and len(mention_map[title][0]) < 3:
            mention_map[title].pop(0)
        mention_map[title] = mention_map[title][:10]
        random.shuffle(mention_map[title])
    mention_map[title] = ' '.join(mention_map[title])

# Analyze examples

The following are good examples:
 - a
 - a
 - a
 - a
 

## Examples of missing mention

In [45]:
sample = df[df['source_title'] == 'FC_Hermannstadt'].sample(1)
target_title = sample['target_title'].values[0]
target_lead = sample['target_lead'].values[0]
first_version = sample['first_version'].values[0]
target_title, target_lead, first_version

('Karlo Letica',
 'Karlo Letica is a Croatian professional footballer who plays as a goalkeeper for Swiss Super League club FC Lausanne-Sport.',
 np.int64(1178541564))

In [37]:
# sample = df[df['missing_category'] == 'missing_sentence'].sample(1)
# sample = df[df['missing_category'].isna()].sample(1)
sample = df[df['source_title'] == 'FC_Hermannstadt'].sample(1)

target_title = sample['target_title'].values[0]
target_lead = sample['target_lead'].values[0]
first_version = sample['first_version'].values[0]

contexts = []
source_sections = []
contexts.append(sample['context'].values[0])
source_sections.append(sample['section'].values[0])
negative_contexts = literal_eval(sample['negative_contexts'].values[0])
for neg_context in negative_contexts:
    contexts.append(neg_context['context'])
    source_sections.append(neg_context['section'])



#print('target_title', target_title)
#print('target_lead', target_lead)
#print('first_version', first_version)
#print('contexts', contexts)
#print('source_sections', source_sections)
#print('number of candidates', len(contexts))

#print('RUNNING MODEL')
best_model, all_scores = model_best_candidate(model, prediction_head, tokenizer, contexts, source_sections, target_title,
                                  target_lead, mention_map)
#print('RESULT', best_model)
#print('SCORES', all_scores)

# Define sample2 with random seed
sample2 = df.sample(n=3, random_state=42)

print(sample2)

# Process each sample in sample2
for i, row in sample2.iterrows():
    print(f"\n--- Processing sample {i} ---")
    
    target_title = row['target_title']
    target_lead = row['target_lead']
    first_version = row['first_version']
    
    contexts = []
    source_sections = []
    
    contexts.append(row['context'])
    source_sections.append(row['section'])
    
    negative_contexts = literal_eval(row['negative_contexts'])
    for neg_context in negative_contexts:
        contexts.append(neg_context['context'])
        source_sections.append(neg_context['section'])
    
    print('target_title:', target_title)
    print('target_lead:', target_lead)
    print('first_version:', first_version)
    #print('contexts:', contexts)
    #print('source_sections:', source_sections)
    #print('number of candidates:', len(contexts))
    
    print('RUNNING MODEL')
    best_model, all_scores = model_best_candidate(model, prediction_head, tokenizer, contexts, source_sections, target_title,
                                      target_lead, mention_map)
    print('RESULT:', best_model)
    print('SCORES:', all_scores)


                              source_title  source_ID   target_title  \
521                 Why_Women_Don%27t_Code   73805935      Quillette   
737             General_Staff_Headquarters   74927680  Republicanism   
740  Danse_Macabre_%28Duran_Duran_album%29   74646240      Pop Trash   

                                               context     section  \
521  "Why Women Don't Code" is an essay by Universi...        Lead   
737  And they massacred Koreans living in the Gando...  Background   
740  2023 studio album by Duran Duran\nDanse Macabr...        Lead   

     mention_present                                        source_lead  \
521            False  "Why Women Don't Code" is an essay by Universi...   
737             True  1924–1929 Korean independence organization\nTh...   
740             True  2023 studio album by Duran Duran\nDanse Macabr...   

     first_version  second_version  direct_match missing_category  \
521     1179676986      1179677123          True  missing_me

In [19]:
%pip instal

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [20]:
import hashlib
import logging
import os

import _md5
import pandas as pd
from dotenv import load_dotenv

from sentence_score import SentenceScorer
from wiki_mongo_db import WikiMongoDB
from wiki_sentence_utils import fetch_article_metadata
from wiki_sentence_utils import parse_article


class WikiPipeline:
    def __init__(self, mongo_uri=None, db_name=None):
        import os
        mongo_uri = mongo_uri or os.environ.get("MONGO_URI", "mongodb://localhost:27017/")
        db_name = db_name or os.environ.get("DB_NAME", "wikinsert")
        # Load environment variables from .env file
        load_dotenv()
        # Paths to input parquet files; change these as needed
        self.source_path = os.environ.get("SOURCE_ARTICLES_PATH", "/path/to/source_articles.parquet")
        self.scored_path = os.environ.get("SCORED_DATA_PATH", "/path/to/score_data.parquet")
        MENTION_MAP_PATH = os.environ.get("MENTION_MAP_PATH", "/path/to/mention_map.parquet")
        MODEL_DIR = os.environ.get("MODEL_DIR", "/path/to/model_dir")
        self.db = WikiMongoDB(mongo_uri, db_name)
        self.scorer = SentenceScorer(MODEL_DIR, MENTION_MAP_PATH)


    def extract_article(self, lang, title, revid):
        """
        1) Check if article (lang, title, revid) is in the DB. If not or if 'force' is True:
           - Parse article
           - Save minimal article metadata to 'articles'
           - Save article sections to 'sections'
        2) Return the article_id (md5).
        """
        article_id = _md5.md5(f"{lang}{title}{revid}".encode()).hexdigest()

        article_meta_data = self.db.get_article_meta_data(article_id)
        if article_meta_data:
            logging.info(f"Article {lang}:{title}:{revid} already in DB (ID={article_id}). Skipping extraction.")
            return article_id

        article_data = parse_article(lang, title, revid)
        article_id = self.db.add_article_meta_data(lang, title, revid, article_data["thumbnail"],
                                                   article_data["description"])
        self.db.add_sections(article_id, article_data["sections"])
        return article_id

    def score_and_store(self, source_id, target_title, target_lead):
        """
        1) Check if there's an existing 'scores' document for (source_article_id, target_title).
        2) If needed, fetch sections of the source article, parse or load the target article to get
           relevant info (like lead section, if desired).
        3) Run scoring, store results in 'scores'.
        """
        # Check if scores already exist
        article_meta = self.db.get_article_meta_data(source_id)
        lang = article_meta["lang"]
        target_metadata = fetch_article_metadata(lang, target_title)

        target_id = self.db.add_target_meta(target_title, lang, target_lead, target_metadata["thumbnail"],
                                            target_metadata["description"])

        pair_id = hashlib.md5(f"{lang}{target_id}{source_id}".encode()).hexdigest()

        existing = self.db.get_scores(pair_id)
        if existing:
            logging.info(f"Scores for pair_id={pair_id} exist. Skipping.")
            return

        # Retrieve source sections:
        source_title = article_meta["title"]

        source_sections = self.db.get_sections(source_id)

        scores = self.scorer.get_sentence_scores(
            source_sections,
            source_title,
            target_title,
            target_lead,
        )
        # Save to DB
        self.db.add_scores(source_id, target_id, scores.to_dict('records'))
        self.db.update_article_targets(source_id, target_title)

    def full_pipeline_example(self):
        """
        Demonstration method:
        1) Extract some articles from a list.
        2) Then score pairs of those articles.
        """
        # Example articles to extract:
        articles_to_extract = [
            ("en", "Salmon", "1264244422"),
            ("en", "Bream", "1276814427"),
        ]
        # Extract each
        extracted_ids = []
        for (lang, title, revid) in articles_to_extract:
            lower_case_title = title.lower()
            article_id = self.extract_article(lang, lower_case_title, revid)
            extracted_ids.append(article_id)

        # Let’s define a few pairs to score
        pairs_to_score = [
            # Pair the first article with the second, for example
            (extracted_ids[1], extracted_ids[0])
        ]

        for (source_id, target_id) in pairs_to_score:
            self.score_and_store(source_id, target_id, )

    def full_pipeline(self, testing):
        """
         Pipeline run:
        - Extract source articles from SOURCE_ARTICLES_PATH
        - Score pairs from SCORED_DATA_PATH
        """
        # Determine language from source file name
        lang = os.path.basename(self.source_path).split('.')[0].split('_')[-1]
        # Load source articles
        df_sources = pd.read_parquet(self.source_path)
        # take only the first 20 for testing
        if testing:
            df_sources = df_sources.head(20)
            df_sources = df.sample(n=3, random_state=42)
        id_map = {}
        for _, row in df_sources.iterrows():
            title = row['source_title'].lower()
            revid = str(row['first_version'])
            src_id = self.extract_article(lang, title, revid)
            id_map[(title, revid)] = src_id
        # Load scoring data
        df_scores = pd.read_parquet(self.scored_path)
        # Process each unique source-target pair
        for _, row in df_scores[
            ['source_title', 'first_version', 'target_title', 'target_lead']].drop_duplicates().iterrows():
            src_title = row['source_title'].lower()
            src_rev = str(row['first_version'])
            src_id = id_map.get((src_title, src_rev))
            if not src_id:
                logging.warning(f"Source {src_title} rev {src_rev} not found; skipping.")
                continue
            tgt_title = row['target_title']
            tgt_lead = row['target_lead']
            self.score_and_store(src_id, tgt_title, tgt_lead)
            logging.info(f"Scored {src_title} (rev {src_rev}) against {tgt_title}.")

def main():
    logging.basicConfig(level=logging.INFO)
    pipeline = WikiPipeline()

    # This main() gives an example pipeline run:
    # pipeline.full_pipeline_example()

    # Uncomment the line below to run the full pipeline with parquet files
    pipeline.full_pipeline(testing=True)
    logging.info("Finished testing")
    # withouth testing
    # pipeline.full_pipeline(testing=False)


if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'pymongo'

In [33]:
df_sources = df.sample(n=3, random_state=42)
df_sources.keys()

Index(['source_title', 'source_ID', 'target_title', 'context', 'section',
       'mention_present', 'source_lead', 'first_version', 'second_version',
       'direct_match', 'missing_category', 'negative_contexts', 'target_lead'],
      dtype='object')

In [34]:
df_sources

,source_title,source_ID,target_title,context,section,mention_present,source_lead,first_version,second_version,direct_match,missing_category,negative_contexts,target_lead
521,Why_Women_Don%27t_Code,73805935,Quillette,"""Why Women Don't Code"" is an essay by Universi...",Lead,False,"""Why Women Don't Code"" is an essay by Universi...",1179676986,1179677123,True,missing_mention,"[{'context': '""Why Women Don\'t Code"" is an es...",Quillette is an online magazine founded by Aus...
737,General_Staff_Headquarters,74927680,Republicanism,And they massacred Koreans living in the Gando...,Background,True,1924–1929 Korean independence organization\nTh...,1179279064,1179288259,True,None,"[{'context': 'The General Staff Headquarters, ...",Republicanism is a political ideology centred ...
740,Danse_Macabre_%28Duran_Duran_album%29,74646240,Pop Trash,2023 studio album by Duran Duran\nDanse Macabr...,Lead,True,2023 studio album by Duran Duran\nDanse Macabr...,1176838804,1177854249,True,None,"[{'context': ""2023 studio album by Duran Duran...",Pop Trash is the tenth studio album by English...


In [32]:
random.seed(42)
random.random()

0.6394267984578837

AttributeError: 'dict' object has no attribute 'size'